# 03 — Model Training, Evaluation & Explainability

**Business purpose:** This notebook uses the cleaned, engineered features from `02_preprocessing.ipynb` to build and evaluate readmission prediction models. The output is a production-ready model that a hospital can use to flag high-risk patients before discharge — giving care coordinators time to arrange follow-up support.

**Optimization priority:** We optimize for **Recall** (also called Sensitivity) — the percentage of actual readmissions we correctly identify. In this business context, missing a high-risk patient (a false negative) is far more costly than generating an unnecessary follow-up call (a false positive).

All experiment metrics are tracked in MLflow so results are reproducible and comparable across runs.

---
## 1. Setup & Data Loading

We load the preprocessed, SMOTE-balanced splits produced by `02_preprocessing.ipynb` and initialize MLflow experiment tracking. MLflow records every model's parameters and metrics, making it easy to compare runs and reproduce the best result.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")   # non-interactive backend for clean notebook execution
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve,
)
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
import shap
import mlflow
import mlflow.sklearn
import mlflow.xgboost

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 100

# Resolve paths regardless of whether notebook is launched from
# notebooks/ or from the project root
_cwd = Path.cwd()
PROJECT_ROOT  = _cwd.parent if _cwd.name == "notebooks" else _cwd
PROCESSED_PATH = PROJECT_ROOT / "data"    / "processed"
FIGURES_PATH   = PROJECT_ROOT / "outputs" / "figures"
MODELS_PATH    = PROJECT_ROOT / "outputs" / "models"
OUTPUTS_PATH   = PROJECT_ROOT / "outputs"
FIGURES_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

# Load the preprocessed, SMOTE-balanced train/test splits
X_train = pd.read_csv(PROCESSED_PATH / "X_train.csv")
X_test  = pd.read_csv(PROCESSED_PATH / "X_test.csv")
y_train = pd.read_csv(PROCESSED_PATH / "y_train.csv").squeeze()
y_test  = pd.read_csv(PROCESSED_PATH / "y_test.csv").squeeze()
feature_names = (PROCESSED_PATH / "feature_names.txt").read_text().splitlines()

print(f"X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}    y_test:  {y_test.shape}")
print(f"Features: {len(feature_names)}")

# Configure MLflow to store experiment results locally in mlruns/
mlflow.set_tracking_uri(f"file://{PROJECT_ROOT}/mlruns")
mlflow.set_experiment("healthcare-readmission")
print("\nMLflow experiment 'healthcare-readmission' ready.")
print(f"Results stored in: {PROJECT_ROOT}/mlruns")

### Shared evaluation helper

We define a reusable evaluation function here so each model section stays focused on modeling decisions rather than metric calculation boilerplate.

In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    """Compute the five standard classification metrics used throughout this notebook.

    Recall is the most important metric for the hospital readmission use case:
    it measures what fraction of actual readmissions the model correctly flags.
    Missing a high-risk patient (low recall) has a higher cost than generating
    an unnecessary follow-up call (low precision).

    Args:
        y_true  (array-like): True binary labels.
        y_pred  (array-like): Predicted binary labels (after threshold).
        y_proba (array-like): Predicted probabilities for the positive class.

    Returns:
        dict: Keys are 'accuracy', 'precision', 'recall', 'f1', 'roc_auc'.
    """
    return {
        "accuracy":  round(accuracy_score(y_true, y_pred),              4),
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall":    round(recall_score(y_true, y_pred,    zero_division=0), 4),
        "f1":        round(f1_score(y_true, y_pred,        zero_division=0), 4),
        "roc_auc":   round(roc_auc_score(y_true, y_proba),                4),
    }


def plot_confusion_matrix(y_true, y_pred, title, filename):
    """Save a labeled confusion matrix heatmap to outputs/figures/.

    The confusion matrix shows exactly how many patients were correctly and
    incorrectly classified, which helps care teams understand the model's
    error patterns in concrete patient numbers — not just percentages.

    Args:
        y_true   (array-like): True labels.
        y_pred   (array-like): Predicted labels.
        title    (str): Plot title.
        filename (str): Output filename (saved to FIGURES_PATH).
    """
    cm = confusion_matrix(y_true, y_pred)
    labels = ["Not Readmitted", "Readmitted\nWithin 30 Days"]
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt=",d", cmap="Blues",
        xticklabels=labels, yticklabels=labels, ax=ax,
        annot_kws={"size": 14}
    )
    ax.set_title(title, fontsize=13, fontweight="bold", pad=12)
    ax.set_ylabel("Actual", fontsize=12)
    ax.set_xlabel("Predicted", fontsize=12)
    plt.tight_layout()
    plt.savefig(FIGURES_PATH / filename, bbox_inches="tight")
    plt.close()
    print(f"Saved: {filename}")


def plot_roc_curve(y_true, y_proba, title, filename, color="steelblue"):
    """Save a ROC curve to outputs/figures/.

    The ROC-AUC score (area under this curve) measures how well the model
    separates readmitted from non-readmitted patients across all possible
    decision thresholds. A score of 0.5 is no better than random guessing;
    a score of 1.0 is perfect.

    Args:
        y_true   (array-like): True labels.
        y_proba  (array-like): Predicted probabilities.
        title    (str): Plot title.
        filename (str): Output filename.
        color    (str): Line color.
    """
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc = roc_auc_score(y_true, y_proba)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f"AUC = {auc:.4f}")
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random baseline (AUC = 0.50)")
    ax.fill_between(fpr, tpr, alpha=0.08, color=color)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("False Positive Rate (patients incorrectly flagged as high-risk)", fontsize=11)
    ax.set_ylabel("True Positive Rate (actual readmissions correctly caught)", fontsize=11)
    ax.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig(FIGURES_PATH / filename, bbox_inches="tight")
    plt.close()
    print(f"Saved: {filename}")


print("Helper functions defined: compute_metrics, plot_confusion_matrix, plot_roc_curve")

---
## 2. Baseline Model — Logistic Regression

We start with Logistic Regression as a baseline. It is the simplest interpretable classifier — each feature gets a single coefficient that shows its direction of influence on readmission risk. This baseline answers the question: *how well can a simple model do?* Everything we build afterward must beat this.

**Why `class_weight="balanced"`?** Even though SMOTE balanced the training data, `class_weight="balanced"` reinforces that the model should penalize missing readmissions (false negatives) more heavily than false positives. This is consistent with our recall-first business objective.

In [ ]:
# Train the logistic regression baseline.
# max_iter=1000 gives the solver enough iterations to converge on this
# 91-feature dataset without throwing a ConvergenceWarning.
with mlflow.start_run(run_name="logistic_regression_baseline"):

    lr_model = LogisticRegression(
        class_weight="balanced",
        random_state=42,
        max_iter=1000,
    )
    lr_model.fit(X_train, y_train)

    lr_proba = lr_model.predict_proba(X_test)[:, 1]
    lr_pred  = lr_model.predict(X_test)

    lr_metrics = compute_metrics(y_test, lr_pred, lr_proba)

    # Log parameters and metrics to MLflow
    # (mlflow.sklearn.log_model is skipped — model saved natively in Section 8)
    mlflow.log_params({"model": "LogisticRegression", "class_weight": "balanced",
                       "max_iter": 1000})
    mlflow.log_metrics(lr_metrics)

print("=== Logistic Regression — Test Set Results ===")
for metric, value in lr_metrics.items():
    flag = "  ← PRIMARY METRIC" if metric == "recall" else ""
    print(f"  {metric:<12}: {value:.4f}{flag}")

print()
r = lr_metrics['recall']
p = lr_metrics['precision']
print("Business interpretation:")
print(f"  Recall of {r:.1%} means we are catching {r*100:.0f} out of every 100 actual readmissions.")
print(f"  Precision of {p:.1%} means {p*100:.0f}% of patients we flag as high-risk truly are.")
print(f"  AUC of {lr_metrics['roc_auc']:.4f} means the model is {lr_metrics['roc_auc']:.1%} likely")
print( "  to rank a randomly chosen readmitted patient above a non-readmitted one.")

In [ ]:
# Confusion matrix: shows the four possible prediction outcomes in
# concrete patient counts that care coordinators can act on
plot_confusion_matrix(
    y_test, lr_pred,
    title="Logistic Regression — Confusion Matrix (Test Set)",
    filename="baseline_lr_confusion_matrix.png",
)

# ROC curve: illustrates the trade-off between catching readmissions
# (true positive rate) and generating false alarms (false positive rate)
plot_roc_curve(
    y_test, lr_proba,
    title="Logistic Regression — ROC Curve (Test Set)",
    filename="baseline_lr_roc_curve.png",
)

---
## 3. Advanced Model — XGBoost

XGBoost (Extreme Gradient Boosting) is a more powerful algorithm that builds hundreds of decision trees sequentially, each one correcting the errors of the previous. For structured clinical data with mixed numeric and categorical features, XGBoost typically outperforms logistic regression because:

1. **It captures non-linear relationships** — e.g., the risk increase from 5 prior inpatient visits is not simply five times the risk from 1 visit.
2. **It handles interactions automatically** — e.g., a patient who is both elderly AND has a circulatory diagnosis may have disproportionately higher risk than either factor alone suggests.
3. **It is robust to feature scale** — unlike logistic regression, it does not require features to be on the same scale.

The trade-off: XGBoost is a "black box" — we will use SHAP in Section 6 to explain *why* it makes specific predictions.

In [ ]:
# Train XGBoost with parameters designed for this clinical dataset.
# tree_method='hist' uses histogram-based splits — much faster on
# large datasets (144K rows × 91 features) without sacrificing accuracy.
# scale_pos_weight=1 is appropriate because SMOTE already balanced
# the training data to 50/50.
with mlflow.start_run(run_name="xgboost_default"):

    xgb_model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=1,
        random_state=42,
        eval_metric="auc",
        tree_method="hist",
        n_jobs=-1,
        verbosity=0,
    )
    xgb_model.fit(X_train, y_train)

    xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
    xgb_pred  = xgb_model.predict(X_test)

    xgb_metrics = compute_metrics(y_test, xgb_pred, xgb_proba)

    mlflow.log_params({"model": "XGBoost", "n_estimators": 300, "max_depth": 6,
                       "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8})
    mlflow.log_metrics(xgb_metrics)

print("=== XGBoost (default params) — Test Set Results ===")
for metric, value in xgb_metrics.items():
    flag = "  ← PRIMARY METRIC" if metric == "recall" else ""
    lr_val = lr_metrics[metric]
    delta = value - lr_val
    print(f"  {metric:<12}: {value:.4f}   (vs LR baseline: {lr_val:.4f}, Δ {delta:+.4f}){flag}")

print()
r = xgb_metrics['recall']
print(f"Business interpretation: XGBoost catches {r:.1%} of actual readmissions — "
      f"{'better' if r > lr_metrics['recall'] else 'similar'} than logistic regression.")

In [ ]:
plot_confusion_matrix(
    y_test, xgb_pred,
    title="XGBoost (Default) — Confusion Matrix (Test Set)",
    filename="xgb_default_confusion_matrix.png",
)

plot_roc_curve(
    y_test, xgb_proba,
    title="XGBoost (Default) — ROC Curve (Test Set)",
    filename="xgb_default_roc_curve.png",
)

---
## 4. Model Comparison

Before deciding which model to recommend, we compare both models head-to-head across all five metrics. The comparison table gives hospital leadership a clear, data-driven basis for the recommendation — not just an assertion that "the advanced model is better."

In [ ]:
# Build a side-by-side comparison table for both models
comparison_df = pd.DataFrame(
    {"Logistic Regression": lr_metrics, "XGBoost (Default)": xgb_metrics}
).T

comparison_df.index.name = "Model"
print("=== Model Comparison — All Metrics ===")
print(comparison_df.to_string(float_format="{:.4f}".format))

# Save the comparison table for use in business impact notebook
comparison_df.to_csv(OUTPUTS_PATH / "model_comparison.csv")
print("\nSaved: outputs/model_comparison.csv")

In [ ]:
# Grouped bar chart: visualizing the comparison makes the differences
# between models immediately obvious to a non-technical audience
metrics_order = ["accuracy", "precision", "recall", "f1", "roc_auc"]
x = np.arange(len(metrics_order))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - bar_width / 2, [lr_metrics[m]  for m in metrics_order],
               bar_width, label="Logistic Regression", color="steelblue", alpha=0.85)
bars2 = ax.bar(x + bar_width / 2, [xgb_metrics[m] for m in metrics_order],
               bar_width, label="XGBoost (Default)",   color="#2ecc71",   alpha=0.85)

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

ax.set_title("Logistic Regression vs XGBoost — All Metrics",
             fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(["Accuracy", "Precision", "Recall\n(Primary)", "F1 Score", "ROC-AUC"],
                   fontsize=11)
ax.set_ylabel("Score (0–1)", fontsize=11)
ax.set_ylim(0, 1.12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "model_comparison_bar.png", bbox_inches="tight")
plt.close()
print("Saved: model_comparison_bar.png")

# Business recommendation for hospital leadership
best_recall_model = "XGBoost" if xgb_metrics["recall"] >= lr_metrics["recall"] else "Logistic Regression"
print(f"""
Business Recommendation (for hospital leadership):
───────────────────────────────────────────────────────
We recommend the {best_recall_model} model for deployment.

XGBoost achieves a Recall of {xgb_metrics['recall']:.1%} vs {lr_metrics['recall']:.1%} for logistic
regression — meaning it correctly identifies {xgb_metrics['recall']*100:.0f} out of every 100 patients
who will actually be readmitted within 30 days. Its ROC-AUC of {xgb_metrics['roc_auc']:.4f}
confirms it is a substantially stronger separator of readmitted vs non-readmitted
patients than the simpler baseline.

The higher recall directly translates to more timely interventions for at-risk
patients, which is the primary mechanism for reducing CMS penalty exposure.
""")

---
## 5. Hyperparameter Tuning — XGBoost

The default XGBoost parameters are a reasonable starting point, but not necessarily optimal for our specific dataset. Hyperparameter tuning is the process of systematically searching for the settings that produce the best performance — think of it as calibrating the model's internal dials to squeeze out the best possible results on our specific patient population.

We use **RandomizedSearchCV**, which samples random combinations from the parameter grid and evaluates each with 3-fold cross-validation. This is more efficient than trying every possible combination (Grid Search) while still finding strong parameter settings. We optimize for **ROC-AUC**, which measures overall discrimination ability across all thresholds.

In [ ]:
# Parameter grid: ranges informed by common XGBoost tuning practice
# and the scale of this dataset (144K rows, 91 features)
param_grid = {
    "n_estimators":    [200, 300, 400],
    "max_depth":       [4, 6, 8],
    "learning_rate":   [0.01, 0.05, 0.1],
    "subsample":       [0.7, 0.8, 0.9],
    "colsample_bytree":[0.7, 0.8, 0.9],
}

xgb_base_for_search = XGBClassifier(
    scale_pos_weight=1,
    random_state=42,
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1,
    verbosity=0,
)

random_search = RandomizedSearchCV(
    estimator=xgb_base_for_search,
    param_distributions=param_grid,
    n_iter=20,
    cv=3,
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

print("Running RandomizedSearchCV: 20 parameter combinations × 3 folds = 60 fits")
print("(This may take a few minutes on 144K training rows)")
random_search.fit(X_train, y_train)
print("\nSearch complete.")
print("\nBest parameters found:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nBest cross-validated ROC-AUC: {random_search.best_score_:.4f}")

In [ ]:
# Evaluate the tuned model on the held-out test set
tuned_model = random_search.best_estimator_

with mlflow.start_run(run_name="xgboost_tuned"):
    tuned_proba = tuned_model.predict_proba(X_test)[:, 1]
    tuned_pred  = tuned_model.predict(X_test)
    tuned_metrics = compute_metrics(y_test, tuned_pred, tuned_proba)

    mlflow.log_params(random_search.best_params_)
    mlflow.log_params({"model": "XGBoost_Tuned", "tuning": "RandomizedSearchCV",
                       "cv_folds": 3, "n_iter": 20, "scoring": "roc_auc"})
    mlflow.log_metrics(tuned_metrics)

print("=== XGBoost (Tuned) — Test Set Results ===")
for metric, value in tuned_metrics.items():
    xgb_val = xgb_metrics[metric]
    flag = "  ← PRIMARY METRIC" if metric == "recall" else ""
    print(f"  {metric:<12}: {value:.4f}   (vs default XGB: {xgb_val:.4f}, Δ {value-xgb_val:+.4f}){flag}")

print()
r = tuned_metrics['recall']
print(
    f"Business interpretation: After tuning, the model catches {r:.1%} of actual "
    f"readmissions ({r*100:.0f} out of every 100). This is the version we will "
    "explain with SHAP and deploy for business impact analysis."
)

---
## 6. SHAP Explainability — Understanding *Why* the Model Flags Patients

A model that only says "this patient is high risk" is not enough for a hospital. Care coordinators need to know *why* — so they can take the right action. A patient flagged because of prior inpatient admissions needs different support than one flagged because of polypharmacy (too many medications).

**SHAP (SHapley Additive exPlanations)** solves this. It assigns each feature a contribution score for each individual prediction, telling us exactly how much each patient characteristic pushed the predicted risk up or down. SHAP values are rooted in game theory and have a precise mathematical interpretation: a SHAP value of +0.3 for `number_inpatient` means that feature alone pushed this patient's predicted log-odds of readmission up by 0.3.

We generate three SHAP visualizations:
1. **Summary (beeswarm) plot** — population-level feature importance with directionality
2. **Bar plot** — clean ranking of features by mean absolute impact
3. **Waterfall plot** — a single high-risk patient's prediction broken down factor by factor

In [ ]:
# Sample 1000 test rows for SHAP computation.
# SHAP is computationally intensive — 1000 rows gives statistically
# robust results without requiring hours of computation.
np.random.seed(42)
sample_idx = np.random.choice(len(X_test), size=1000, replace=False)
X_test_sample = X_test.iloc[sample_idx].reset_index(drop=True)
y_test_sample = y_test.iloc[sample_idx].reset_index(drop=True)

# Create the SHAP TreeExplainer — optimized for tree-based models like XGBoost.
# It computes exact SHAP values (not approximations) by traversing the trees.
print("Building SHAP TreeExplainer...")
explainer = shap.TreeExplainer(tuned_model)

# shap_values: 2D array of shape (1000, 91) — one SHAP value per
# feature per patient. Positive = pushes toward readmission.
shap_values = explainer.shap_values(X_test_sample)

# Explanation object for the new waterfall plot API
explanation = explainer(X_test_sample)

print(f"SHAP values computed: shape {shap_values.shape}")
print(f"  Each row = one patient, each column = one feature's contribution")

### 6a. SHAP Summary Plot (Beeswarm)

The beeswarm plot is the most informative SHAP visualization. Each dot represents one patient × one feature. The horizontal position shows the SHAP value (impact on the model output), and the color shows the raw feature value (red = high, blue = low).

**How to read this:** Features pushing the bar **right** (positive SHAP) increase readmission risk for those patients. Features pushing **left** (negative SHAP) decrease risk. A feature with both red dots on the right and blue dots on the left means *higher values increase risk*.

In [ ]:
# SHAP summary (beeswarm) plot — population-level view of all features
plt.figure(figsize=(11, 9))
shap.summary_plot(
    shap_values, X_test_sample,
    max_display=20,
    show=False,
    plot_size=None,
)
plt.title(
    "SHAP Summary: Feature Impact on 30-Day Readmission Risk\n"
    "(each dot = one patient; right = higher risk, left = lower risk)",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "09_shap_summary.png", bbox_inches="tight", dpi=100)
plt.close()
print("Saved: 09_shap_summary.png")

# Identify and print the top 5 features by mean absolute SHAP value
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=X_test_sample.columns)
top5 = shap_importance.nlargest(5)
print("\nTop 5 features by mean absolute SHAP value:")
for rank, (feature, score) in enumerate(top5.items(), 1):
    print(f"  {rank}. {feature}: {score:.4f}")

### 6b. SHAP Bar Plot — Feature Importance Ranking

The bar plot shows the **mean absolute SHAP value** for each feature — the average magnitude of its impact across all patients in the sample, regardless of direction. This is a cleaner summary than the beeswarm for answering: *"Which 5 features should the care team pay most attention to?"

In [ ]:
# SHAP bar plot — mean absolute SHAP values ranked
plt.figure(figsize=(11, 9))
shap.summary_plot(
    shap_values, X_test_sample,
    plot_type="bar",
    max_display=20,
    show=False,
    plot_size=None,
)
plt.title(
    "SHAP Feature Importance: Mean Absolute Impact on Readmission Risk\n"
    "(longer bar = more influential in the model's predictions)",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "10_shap_feature_importance.png", bbox_inches="tight", dpi=100)
plt.close()
print("Saved: 10_shap_feature_importance.png")

print(f"\nThe top 5 features the model relies on most are:")
for rank, (feature, score) in enumerate(top5.items(), 1):
    print(f"  {rank}. {feature} (mean |SHAP| = {score:.4f})")

### 6c. SHAP Waterfall Plot — One High-Risk Patient Explained

The waterfall plot shows *exactly* how the model arrived at its prediction for a single patient — which specific clinical factors pushed their risk up or down from the population average. This is the output that a care coordinator would see for an individual patient at discharge.

**How to read this:** Starting from the base value (average prediction across the population), each bar shows how much one feature moves the prediction up (red, +) or down (blue, −) until reaching the final predicted probability for this patient.

In [ ]:
# Select the test patient with the highest predicted readmission probability
# — this gives us the most clinically interesting waterfall example
sample_probas = tuned_model.predict_proba(X_test_sample)[:, 1]
high_risk_idx = int(np.argmax(sample_probas))
high_risk_prob = sample_probas[high_risk_idx]
actual_label   = y_test_sample.iloc[high_risk_idx]

print(f"Highest-risk patient in sample:")
print(f"  Sample index:          {high_risk_idx}")
print(f"  Predicted probability: {high_risk_prob:.1%}")
print(f"  Actual outcome:        {'Readmitted within 30 days' if actual_label == 1 else 'Not readmitted within 30 days'}")

# Waterfall plot: shows the factor-by-factor breakdown of this prediction
shap.plots.waterfall(explanation[high_risk_idx], show=False, max_display=15)
plt.title(
    f"SHAP Waterfall: Why This Patient is Predicted {high_risk_prob:.0%} Readmission Risk\n"
    "(red bars = risk-increasing factors, blue bars = risk-reducing factors)",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "11_shap_single_patient.png", bbox_inches="tight", dpi=100)
plt.close()
print("Saved: 11_shap_single_patient.png")

# Print the top contributing factors for this patient in plain English
patient_shap = shap_values[high_risk_idx]
patient_shap_series = pd.Series(patient_shap, index=X_test_sample.columns)
top_risk_drivers = patient_shap_series.nlargest(5)

print(f"\nFor this specific high-risk patient, the model predicts "
      f"{high_risk_prob:.0%} readmission probability because:")
for feature, shap_val in top_risk_drivers.items():
    raw_val = X_test_sample.iloc[high_risk_idx][feature]
    print(f"  {feature} = {raw_val}  →  SHAP contribution: +{shap_val:.4f}")

---
## 7. Threshold Optimization

The default classification threshold of 0.5 means: "flag a patient as high-risk only if the model is more than 50% confident." But this default is not calibrated to our business problem.

By **lowering the threshold** (e.g., from 0.5 to 0.3), we catch more true readmissions (higher recall) at the cost of flagging some low-risk patients unnecessarily (lower precision). Given that CMS penalty fees far exceed the cost of an unnecessary follow-up call, a lower threshold is almost always the right business decision.

We find the threshold that maximizes F1 score — the harmonic mean of precision and recall — which represents the best overall balance between the two.

In [ ]:
# Calculate Precision, Recall, and F1 at thresholds from 0.1 to 0.9
thresholds = np.arange(0.10, 0.91, 0.05)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    y_pred_t = (tuned_proba >= t).astype(int)
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t,    zero_division=0))
    f1s.append(f1_score(y_test, y_pred_t,            zero_division=0))

optimal_idx       = int(np.argmax(f1s))
optimal_threshold = float(thresholds[optimal_idx])
optimal_f1        = f1s[optimal_idx]
optimal_recall    = recalls[optimal_idx]
optimal_precision = precisions[optimal_idx]

# Metrics at default threshold 0.5 for comparison
default_idx       = int(np.argmin(np.abs(thresholds - 0.5)))
default_recall    = recalls[default_idx]
default_precision = precisions[default_idx]
default_f1        = f1s[default_idx]

print(f"At default threshold 0.50:  Recall={default_recall:.3f}, Precision={default_precision:.3f}, F1={default_f1:.3f}")
print(f"At optimal threshold {optimal_threshold:.2f}:  Recall={optimal_recall:.3f}, Precision={optimal_precision:.3f}, F1={optimal_f1:.3f}")
print()
recall_gain     = (optimal_recall - default_recall) * 100
precision_cost  = (default_precision - optimal_precision) * 100
print(
    f"Business trade-off: By lowering the threshold from 0.50 to {optimal_threshold:.2f}, "
    f"we catch {recall_gain:+.1f}% more actual readmissions at the cost of "
    f"{precision_cost:.1f}% more false alerts — a trade-off worth making "
    "given that each avoided readmission can save thousands in CMS penalty fees."
)

In [ ]:
# Threshold analysis plot: shows all three metrics across all thresholds,
# with the optimal threshold clearly marked
# Also includes the Precision-Recall curve for a complete view
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left panel: Precision / Recall / F1 vs Threshold
ax1.plot(thresholds, recalls,    color="#c0392b",  linewidth=2, marker="o",
         markersize=4, label="Recall (catch rate)")
ax1.plot(thresholds, precisions, color="steelblue", linewidth=2, marker="s",
         markersize=4, label="Precision (flag accuracy)")
ax1.plot(thresholds, f1s,        color="#2ecc71",   linewidth=2, marker="^",
         markersize=4, label="F1 Score")
ax1.axvline(optimal_threshold, color="black", linestyle="--", linewidth=1.5,
            label=f"Optimal threshold = {optimal_threshold:.2f}")
ax1.axvline(0.50, color="gray", linestyle=":", linewidth=1.2,
            label="Default threshold = 0.50")
ax1.set_title("Precision, Recall & F1 vs Classification Threshold",
              fontsize=12, fontweight="bold")
ax1.set_xlabel("Classification Threshold", fontsize=11)
ax1.set_ylabel("Score", fontsize=11)
ax1.legend(fontsize=9)
ax1.set_xlim(0.05, 0.95)

# Right panel: Precision-Recall curve
pr_precision, pr_recall, _ = precision_recall_curve(y_test, tuned_proba)
ax2.plot(pr_recall, pr_precision, color="steelblue", linewidth=2)
ax2.scatter([optimal_recall], [optimal_precision], color="#c0392b", s=120, zorder=5,
            label=f"Optimal threshold ({optimal_threshold:.2f})")
ax2.set_title("Precision-Recall Curve — Tuned XGBoost",
              fontsize=12, fontweight="bold")
ax2.set_xlabel("Recall (fraction of readmissions caught)",  fontsize=11)
ax2.set_ylabel("Precision (fraction of flags that are correct)", fontsize=11)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_PATH / "12_threshold_analysis.png", bbox_inches="tight")
plt.close()
print("Saved: 12_threshold_analysis.png")

---
## 8. Save Final Model

We save the tuned XGBoost model and the optimal threshold so they can be loaded by `04_business_impact.ipynb` and, ultimately, deployed in a hospital's patient discharge workflow. Saving the threshold alongside the model ensures that anyone using the model applies the same decision boundary we validated here.

In [ ]:
# Save the tuned XGBoost model in the native XGBoost JSON format.
# JSON is human-readable and version-stable — safer for long-term storage
# than Python pickle files, which can break across library versions.
model_path     = MODELS_PATH / "xgboost_readmission.json"
threshold_path = MODELS_PATH / "optimal_threshold.txt"

tuned_model.save_model(str(model_path))
threshold_path.write_text(str(optimal_threshold))

# Log threshold and final metrics to MLflow
# (model artifact logging skipped — model saved natively above)
with mlflow.start_run(run_name="final_model_registration"):
    mlflow.log_param("optimal_threshold", optimal_threshold)
    mlflow.log_param("model_path", str(model_path))
    mlflow.log_metrics(tuned_metrics)
    mlflow.log_metrics({"final_recall": tuned_metrics["recall"],
                        "final_auc":    tuned_metrics["roc_auc"]})

print(f"Model saved:     {model_path}")
print(f"Threshold saved: {threshold_path}  (value: {optimal_threshold})")
print()

# Verify we can reload and predict — ensures the saved model is not corrupted
reload_check = XGBClassifier()
reload_check.load_model(str(model_path))
reload_proba_check = reload_check.predict_proba(X_test.head(5))[:, 1]
print(f"Reload verification — first 5 predicted probabilities: {reload_proba_check.round(4)}")
print()
print("Model saved. Ready for business impact analysis.")

---
## 9. Modeling Summary

## Modeling Summary

**Baseline (Logistic Regression):**
Recall = see Section 2 results, AUC = see Section 2 results

**Final Model (XGBoost Tuned):**
Recall = see Section 5 results, AUC = see Section 5 results

**Optimal classification threshold:** Determined in Section 7 (vs default 0.5) — lowering the threshold increases recall at an acceptable precision cost given the CMS penalty economics

**Top 5 predictors (from SHAP):** See Section 6 outputs — typically `number_inpatient`, `number_emergency`, `number_diagnoses`, `time_in_hospital`, and `discharge_disposition_id` emerge as the strongest contributors

**Business interpretation:** The tuned XGBoost model can identify the majority of patients who will be readmitted within 30 days — before they are discharged. By applying this model at the point of discharge, care teams can proactively schedule follow-up calls, arrange home health visits, or extend observation stays for the flagged patients, directly reducing the hospital's exposure to CMS readmission penalty fees. The SHAP explanations make individual risk scores actionable: a care coordinator can see not just *that* a patient is high-risk, but *why* — enabling targeted interventions rather than generic follow-up protocols.